# 01 Exploring My Google Maps Data

This notebook explores the raw Google Maps Takeout data and establishes the baseline dataset used throughout the project.

The goal is to understand what information Google provides, assess data quality, and prepare the dataset for enrichment and behavioral analysis.

## Loading the Data

The cleaned dataset combines three Google Maps lists: CDMX, London, and Wanna go.

In [5]:
import pandas as pd

places = pd.read_csv("data/processed/master_places.csv")

places.head()

,title,note,url,tags,comment,source_list
0,The Lamb,NaN,https://www.google.com/maps/place/The+Lamb/dat...,NaN,NaN,CDMX
1,La Once Mil - Roma Norte,NaN,https://www.google.com/maps/place/La+Once+Mil+...,NaN,NaN,CDMX
2,Panadería Rosetta,NaN,https://www.google.com/maps/place/Panader%C3%A...,NaN,NaN,CDMX
3,Bagels Lepu,Bagels buenos y grandes,https://www.google.com/maps/place/Bagels+Lepu/...,NaN,NaN,CDMX
4,La Americana,NaN,https://www.google.com/maps/place/La+Americana...,NaN,NaN,CDMX


## Dataset Overview

First, I check the size of the dataset and the distribution of saved places across lists.

In [6]:
print(f"Total saves: {len(places)}")
print(f"Total columns: {places.shape[1]}")

places.columns.tolist()

Total saves: 947
Total columns: 6


['title', 'note', 'url', 'tags', 'comment', 'source_list']

## Visited vs. Wanna Go

The lists also provide a useful behavioral distinction.

Places saved under CDMX and London represent places I have already visited, while the Wanna go list represents places I am interested in visiting.

This allows the project to compare **observed preferences** with **aspirational preferences**.

In [16]:
places["status"].value_counts()

status
wanna_go    647
visited     300
Name: count, dtype: int64

## Data Completeness

Before enriching the dataset, I evaluate how much information is actually available in the original Google Maps export.

In [17]:
completeness = pd.DataFrame({
    "available": places.notna().sum(),
    "missing": places.isna().sum()
})

completeness["available_pct"] = (
    completeness["available"] / len(places) * 100
).round(1)

completeness

,available,missing,available_pct
title,947,0,100.0
note,182,765,19.2
url,947,0,100.0
tags,0,947,0.0
comment,0,947,0.0
source_list,947,0,100.0
status,947,0,100.0


### Initial Finding

The Takeout export contains very little structured information about each place.

While every save includes a place name and Google Maps URL, most contextual information is missing. Only ~19% of places contain a personal note, while tags and comments are entirely empty.

This makes external enrichment a necessary part of the project.

## Enriched dataset exploration

In [21]:
import pandas as pd

places = pd.read_csv("data/processed/master_places_enriched.csv")

print(f"Total places: {len(places)}")
places.head()

Total places: 947


,title,note,url,tags,comment,source_list,status,google_place_id,google_name,google_maps_uri,...,postal_code,latitude,longitude,primary_type,google_types,business_status,google_rating,review_count,price_level,match_status
0,The Lamb,NaN,https://www.google.com/maps/place/The+Lamb/dat...,NaN,NaN,CDMX,visited,ChIJAQAMYTr_0YURqvRwSWD6y4s,The Lamb,https://maps.google.com/?cid=10073420283000190...,...,06700,19.419080,-99.159648,british_restaurant,"british_restaurant, restaurant, food, point_of...",CLOSED_TEMPORARILY,4.5,240.0,NaN,RESOLVED_FROM_URL
1,La Once Mil - Roma Norte,NaN,https://www.google.com/maps/place/La+Once+Mil+...,NaN,NaN,CDMX,visited,ChIJ07ffWgD_0YURMOk1QAszUtY,La Once Mil - Roma Norte,https://maps.google.com/?cid=15443462195621783...,...,06700,19.419951,-99.160306,mexican_restaurant,"mexican_restaurant, restaurant, food, point_of...",OPERATIONAL,4.8,4532.0,PRICE_LEVEL_EXPENSIVE,RESOLVED_FROM_URL
2,Panadería Rosetta,NaN,https://www.google.com/maps/place/Panader%C3%A...,NaN,NaN,CDMX,visited,ChIJ-_GNczr_0YURLJgHYS8ZPMg,Panadería Rosetta,https://maps.google.com/?cid=14428434997470271...,...,06700,19.419888,-99.160540,bakery,"bakery, food_store, store, food, point_of_inte...",OPERATIONAL,4.5,10656.0,PRICE_LEVEL_MODERATE,RESOLVED_FROM_URL
3,Bagels Lepu,Bagels buenos y grandes,https://www.google.com/maps/place/Bagels+Lepu/...,NaN,NaN,CDMX,visited,ChIJX3bLZQD_0YUR7J9pFfGGqPg,Bagels Lepu,https://maps.google.com/?cid=17917719487498002...,...,06700,19.412257,-99.157205,restaurant,"restaurant, bagel_shop, breakfast_restaurant, ...",OPERATIONAL,4.7,1039.0,PRICE_LEVEL_MODERATE,RESOLVED_FROM_URL
4,La Americana,NaN,https://www.google.com/maps/place/La+Americana...,NaN,NaN,CDMX,visited,ChIJ41PHOwD_0YURh4MyIMYytXY,La Americana,https://maps.google.com/?cid=85537988937852199...,...,06100,19.409611,-99.169081,book_store,"book_store, store, point_of_interest, establis...",OPERATIONAL,4.1,267.0,NaN,RESOLVED_FROM_URL


In [23]:
print("VISITED VS WANNA GO")
print(places["status"].value_counts())

print("\nTOP COUNTRIES")
print(places["country"].value_counts().head(15))

print("\nTOP CITIES")
print(places["city"].value_counts().head(20))

print("\nTOP NEIGHBORHOODS")
print(places["neighborhood"].value_counts().head(20))

print("\nTOP PLACE TYPES")
print(places["primary_type"].value_counts().head(30))

print("\nBUSINESS STATUS")
print(places["business_status"].value_counts(dropna=False))

print("\nPRICE LEVEL")
print(places["price_level"].value_counts(dropna=False))

VISITED VS WANNA GO
status
wanna_go    647
visited     300
Name: count, dtype: int64

TOP COUNTRIES
country
United Kingdom    677
Mexico            251
Italy              10
France              7
Colombia            1
Name: count, dtype: int64

TOP CITIES
city
Greater London         588
Ciudad de México       245
Edinburgh               49
Glasgow City            14
Milano                  10
Fife                     8
Paris                    7
Cuauhtémoc               3
Perth and Kinross        3
Highland Council         2
Cartagena de Indias      1
Juárez                   1
Berkshire                1
Kent                     1
Puerto Escondido         1
Centro                   1
Oxfordshire              1
Surrey                   1
Somerset                 1
Rosemarkie               1
Name: count, dtype: int64

TOP NEIGHBORHOODS
neighborhood
Roma Norte                          80
Colonia Condesa                     29
Juárez                              28
Hipódromo               

## Place categories

Google Places provides granular place types. For analysis, these are grouped into broader Carlife categories while preserving the original Google type.

In [26]:
def categorize_place(place_type):

    if pd.isna(place_type):
        return "Other"

    # Restaurants & food
    if (
        "restaurant" in place_type
        or place_type in [
            "pizza_restaurant",
            "taco_restaurant",
            "sushi_restaurant",
            "sandwich_shop",
            "hamburger_restaurant",
            "bagel_shop",
            "meal_takeaway",
            "bar_and_grill",
            "deli",
            "bistro",
            "steak_house",
            "food_court",
        ]
    ):
        return "Restaurant"

    # Coffee, bakeries & tea
    if place_type in [
        "coffee_shop",
        "cafe",
        "cafeteria",
        "bakery",
        "coffee_roastery",
        "coffee_stand",
        "tea_house",
    ]:
        return "Coffee & Bakery"

    # Drinks & nightlife
    if place_type in [
        "pub",
        "irish_pub",
        "bar",
        "gastropub",
        "wine_bar",
        "cocktail_bar",
        "night_club",
    ]:
        return "Drinks & Nightlife"

    # Dessert
    if place_type in [
        "ice_cream_shop",
        "dessert_shop",
        "dessert_restaurant",
    ]:
        return "Dessert"

    # Food shopping
    if place_type in [
        "food_store",
        "grocery_store",
        "supermarket",
        "market",
    ]:
        return "Food Shopping"

    # Culture & things to do
    if place_type in [
        "museum",
        "art_museum",
        "art_gallery",
        "performing_arts_theater",
        "movie_theater",
        "tourist_attraction",
        "castle",
        "historical_landmark",
        "library",
        "live_music_venue",
        "hindu_temple",
    ]:
        return "Culture & Things to Do"

    # Outdoors
    if place_type in [
        "park",
        "national_park",
        "garden",
        "mountain_peak",
        "scenic_spot",
    ]:
        return "Outdoors"

    # Wellness & sports
    if place_type in [
        "gym",
        "spa",
        "yoga_studio",
        "fitness_center",
        "sports_complex",
        "sports_school",
        "nail_salon",
    ]:
        return "Wellness"

    # Hotels
    if place_type in [
        "hotel",
        "lodging",
    ]:
        return "Hotel"

    # Shopping
    if (
        "store" in place_type
        or "shop" in place_type
        or place_type == "garden_center"
    ):
        return "Shopping"

    return "Other"


places["category"] = places[
    "primary_type"
].apply(categorize_place)

places["category"].value_counts()

category
Restaurant                527
Drinks & Nightlife        190
Coffee & Bakery           135
Shopping                   23
Culture & Things to Do     21
Food Shopping              11
Other                      11
Outdoors                    9
Wellness                    7
Dessert                     7
Hotel                       6
Name: count, dtype: int64

## Other cathegory review

In [28]:
places.loc[
    places["category"] == "Other",
    [
        "title",
        "primary_type",
        "google_types",
        "city",
        "source_list"
    ]
]

,title,primary_type,google_types,city,source_list
310,Difane,service,"service, point_of_interest, establishment",Ciudad de México,Wanna go
376,Kotsu,NaN,"point_of_interest, establishment",Ciudad de México,Wanna go
389,Av. Clavería 157,premise,"premise, street_address",Ciudad de México,Wanna go
530,Tower House,premise,"premise, street_address",Greater London,Wanna go
676,The Broad Walk,NaN,route,Greater London,Wanna go
685,Vennel,NaN,route,Edinburgh,Wanna go
700,Swanston Farm,farm,"farm, point_of_interest, service, establishment",Edinburgh,Wanna go
701,The Glenmorangie Distillery Co,manufacturer,"tourist_attraction, manufacturer, point_of_int...",Highland Council,Wanna go
731,Victoria St,NaN,route,Edinburgh,Wanna go
812,Ham Yard,NaN,route,Greater London,Wanna go


## Food & drink analysis

Explore the characteristics of saved food and drink places and compare visited places with places saved for the future.

In [30]:
places = pd.read_csv(
    "data/processed/places_final.csv"
)

print(f"Total places: {len(places)}")
print(f"Total columns: {len(places.columns)}")

places.head()

Total places: 947
Total columns: 27


,title,note,url,tags,comment,source_list,status,google_place_id,google_name,google_maps_uri,...,latitude,longitude,primary_type,google_types,business_status,google_rating,review_count,price_level,match_status,category
0,The Lamb,NaN,https://www.google.com/maps/place/The+Lamb/dat...,NaN,NaN,CDMX,visited,ChIJAQAMYTr_0YURqvRwSWD6y4s,The Lamb,https://maps.google.com/?cid=10073420283000190...,...,19.419080,-99.159648,british_restaurant,"british_restaurant, restaurant, food, point_of...",CLOSED_TEMPORARILY,4.5,240.0,NaN,RESOLVED_FROM_URL,Restaurant
1,La Once Mil - Roma Norte,NaN,https://www.google.com/maps/place/La+Once+Mil+...,NaN,NaN,CDMX,visited,ChIJ07ffWgD_0YURMOk1QAszUtY,La Once Mil - Roma Norte,https://maps.google.com/?cid=15443462195621783...,...,19.419951,-99.160306,mexican_restaurant,"mexican_restaurant, restaurant, food, point_of...",OPERATIONAL,4.8,4532.0,PRICE_LEVEL_EXPENSIVE,RESOLVED_FROM_URL,Restaurant
2,Panadería Rosetta,NaN,https://www.google.com/maps/place/Panader%C3%A...,NaN,NaN,CDMX,visited,ChIJ-_GNczr_0YURLJgHYS8ZPMg,Panadería Rosetta,https://maps.google.com/?cid=14428434997470271...,...,19.419888,-99.160540,bakery,"bakery, food_store, store, food, point_of_inte...",OPERATIONAL,4.5,10656.0,PRICE_LEVEL_MODERATE,RESOLVED_FROM_URL,Coffee & Bakery
3,Bagels Lepu,Bagels buenos y grandes,https://www.google.com/maps/place/Bagels+Lepu/...,NaN,NaN,CDMX,visited,ChIJX3bLZQD_0YUR7J9pFfGGqPg,Bagels Lepu,https://maps.google.com/?cid=17917719487498002...,...,19.412257,-99.157205,restaurant,"restaurant, bagel_shop, breakfast_restaurant, ...",OPERATIONAL,4.7,1039.0,PRICE_LEVEL_MODERATE,RESOLVED_FROM_URL,Restaurant
4,La Americana,NaN,https://www.google.com/maps/place/La+Americana...,NaN,NaN,CDMX,visited,ChIJ41PHOwD_0YURh4MyIMYytXY,La Americana,https://maps.google.com/?cid=85537988937852199...,...,19.409611,-99.169081,book_store,"book_store, store, point_of_interest, establis...",OPERATIONAL,4.1,267.0,NaN,RESOLVED_FROM_URL,Shopping


In [31]:
food_categories = [
    "Restaurant",
    "Drinks & Nightlife",
    "Coffee & Bakery",
    "Dessert",
    "Food Shopping"
]

food = places[
    places["category"].isin(food_categories)
].copy()

print(f"Food & drink places: {len(food)}")
print(f"Share of all places: {len(food) / len(places):.1%}")

print("\nBY CATEGORY")
print(food["category"].value_counts())

print("\nVISITED VS WANNA GO")
print(food["status"].value_counts())

Food & drink places: 870
Share of all places: 91.9%

BY CATEGORY
category
Restaurant            527
Drinks & Nightlife    190
Coffee & Bakery       135
Food Shopping          11
Dessert                 7
Name: count, dtype: int64

VISITED VS WANNA GO
status
wanna_go    585
visited     285
Name: count, dtype: int64


### Google place types

Google Places can assign multiple types to each place. These provide more detail than the primary type alone and can be used to derive cuisine and format.

In [32]:
food[
    [
        "title",
        "primary_type",
        "google_types",
        "status"
    ]
].sample(
    30,
    random_state=42
)

,title,primary_type,google_types,status
421,Jenni’s Quesadillas,mexican_restaurant,"mexican_restaurant, restaurant, food, point_of...",wanna_go
72,Ginza Barra,japanese_restaurant,"japanese_restaurant, sushi_restaurant, restaur...",visited
528,The Rose & Crown,pub,"pub, bar, restaurant, food, point_of_interest,...",wanna_go
73,Eno,restaurant,"restaurant, breakfast_restaurant, coffee_shop,...",visited
932,The Ship,pub,"pub, beer_garden, sports_bar, bar, restaurant,...",wanna_go
836,Jolene Colebrooke Row,bakery,"bakery, food_store, food, store, point_of_inte...",wanna_go
92,La Bonvi - Virreyes,taco_restaurant,"taco_restaurant, restaurant, food, point_of_in...",visited
877,Sophie's Soho,steak_house,"steak_house, bar, restaurant, food, point_of_i...",wanna_go
647,All My Friends,cocktail_bar,"cocktail_bar, night_club, bar, restaurant, foo...",wanna_go
497,DALLA,italian_restaurant,"italian_restaurant, restaurant, food, point_of...",wanna_go


### Available Google types

Explore all Google place types associated with food and drink locations before deriving cuisine and format features.

In [34]:
all_types = (
    food["google_types"]
    .dropna()
    .str.split(", ")
    .explode()
    .value_counts()
)

print(f"Unique Google types: {len(all_types)}")

all_types.head(100)

for place_type, count in all_types.items():
    print(f"{place_type}: {count}")

Unique Google types: 144
point_of_interest: 870
establishment: 870
food: 790
restaurant: 668
bar: 295
store: 181
food_store: 161
cafe: 142
pub: 121
coffee_shop: 97
bakery: 60
cocktail_bar: 59
italian_restaurant: 58
wine_bar: 57
japanese_restaurant: 43
mexican_restaurant: 40
gastropub: 35
breakfast_restaurant: 35
event_venue: 35
meal_takeaway: 33
mediterranean_restaurant: 33
brunch_restaurant: 33
british_restaurant: 31
pizza_restaurant: 27
seafood_restaurant: 27
french_restaurant: 27
sushi_restaurant: 25
fine_dining_restaurant: 20
taco_restaurant: 20
service: 20
liquor_store: 17
confectionery: 17
european_restaurant: 16
dessert_shop: 16
spanish_restaurant: 15
sandwich_shop: 14
american_restaurant: 14
vegetarian_restaurant: 14
night_club: 13
beer_garden: 12
live_music_venue: 12
steak_house: 11
indian_restaurant: 11
chinese_restaurant: 11
hamburger_restaurant: 11
asian_restaurant: 10
thai_restaurant: 10
bagel_shop: 9
food_delivery: 9
manufacturer: 9
greek_restaurant: 9
grocery_store: 9
bi

### Cuisine

Cuisine is derived from Google place types. A place may have more than one cuisine when Google assigns multiple cuisine-specific types.

In [35]:
cuisine_map = {
    "italian_restaurant": "Italian",
    "japanese_restaurant": "Japanese",
    "mexican_restaurant": "Mexican",
    "mediterranean_restaurant": "Mediterranean",
    "british_restaurant": "British",
    "french_restaurant": "French",
    "spanish_restaurant": "Spanish",
    "american_restaurant": "American",
    "indian_restaurant": "Indian",
    "chinese_restaurant": "Chinese",
    "asian_restaurant": "Asian",
    "thai_restaurant": "Thai",
    "greek_restaurant": "Greek",
    "lebanese_restaurant": "Lebanese",
    "australian_restaurant": "Australian",
    "middle_eastern_restaurant": "Middle Eastern",
    "basque_restaurant": "Basque",
    "latin_american_restaurant": "Latin American",
    "turkish_restaurant": "Turkish",
    "korean_restaurant": "Korean",
    "cantonese_restaurant": "Cantonese",
    "austrian_restaurant": "Austrian",
    "filipino_restaurant": "Filipino",
    "eastern_european_restaurant": "Eastern European",
    "persian_restaurant": "Persian",
    "peruvian_restaurant": "Peruvian",
    "polish_restaurant": "Polish",
    "german_restaurant": "German",
    "vietnamese_restaurant": "Vietnamese",
    "brazilian_restaurant": "Brazilian",
    "cajun_restaurant": "Cajun",
    "south_indian_restaurant": "South Indian",
    "caribbean_restaurant": "Caribbean",
    "south_american_restaurant": "South American",
    "western_restaurant": "Western",
}


def get_cuisines(types):

    if pd.isna(types):
        return None

    place_types = types.split(", ")

    cuisines = [
        cuisine_map[t]
        for t in place_types
        if t in cuisine_map
    ]

    if not cuisines:
        return None

    return ", ".join(dict.fromkeys(cuisines))


food["cuisine"] = food[
    "google_types"
].apply(get_cuisines)

food["cuisine"].value_counts().head(30)

cuisine
Italian                          51
Mexican                          39
Japanese                         39
British                          27
French                           20
Mediterranean                    15
American                         12
Indian                           10
Thai                             10
Chinese                           7
Spanish                           7
Greek                             6
Italian, Mediterranean            4
Australian                        4
Lebanese                          3
Asian                             3
Korean                            2
Austrian                          2
Asian, Japanese                   2
Cantonese, Chinese                2
French, Mediterranean             2
Persian                           2
Spanish, Mediterranean            2
Eastern European                  2
Basque, Spanish                   2
Filipino                          2
Cajun, American                   1
Vietnamese          

### Format

Format describes how a place is experienced rather than the cuisine it serves.

In [36]:
format_map = {
    "restaurant": "Restaurant",
    "fine_dining_restaurant": "Fine Dining",
    "bistro": "Bistro",
    "gastropub": "Gastropub",
    "pub": "Pub",
    "irish_pub": "Pub",
    "bar": "Bar",
    "cocktail_bar": "Cocktail Bar",
    "wine_bar": "Wine Bar",
    "lounge_bar": "Lounge Bar",
    "sports_bar": "Sports Bar",
    "beer_garden": "Beer Garden",
    "brewpub": "Brewpub",

    "coffee_shop": "Coffee Shop",
    "cafe": "Cafe",
    "coffee_roastery": "Coffee Roastery",
    "coffee_stand": "Coffee Stand",
    "tea_house": "Tea House",

    "bakery": "Bakery",
    "pastry_shop": "Pastry Shop",
    "cake_shop": "Cake Shop",
    "bagel_shop": "Bagels",
    "deli": "Deli",

    "pizza_restaurant": "Pizza",
    "taco_restaurant": "Tacos",
    "sushi_restaurant": "Sushi",
    "hamburger_restaurant": "Burgers",
    "steak_house": "Steakhouse",
    "seafood_restaurant": "Seafood",
    "oyster_bar_restaurant": "Oyster Bar",
    "tapas_restaurant": "Tapas",
    "ramen_restaurant": "Ramen",
    "noodle_shop": "Noodles",
    "dumpling_restaurant": "Dumplings",
    "falafel_restaurant": "Falafel",
    "fish_and_chips_restaurant": "Fish & Chips",
    "barbecue_restaurant": "BBQ",

    "breakfast_restaurant": "Breakfast",
    "brunch_restaurant": "Brunch",
    "sandwich_shop": "Sandwiches",
    "ice_cream_shop": "Ice Cream",
    "dessert_shop": "Dessert",
    "juice_shop": "Juice",

    "meal_takeaway": "Takeaway",
    "fast_food_restaurant": "Fast Food",
    "food_court": "Food Court",
}


def get_formats(types):

    if pd.isna(types):
        return None

    place_types = types.split(", ")

    formats = [
        format_map[t]
        for t in place_types
        if t in format_map
    ]

    if not formats:
        return None

    return ", ".join(dict.fromkeys(formats))


food["format"] = food[
    "google_types"
].apply(get_formats)

food["format"].value_counts().head(30)

format
Restaurant                                 266
Coffee Shop, Cafe                           45
Pub, Bar                                    42
Pub, Bar, Restaurant                        30
Bar, Restaurant                             23
Bakery                                      21
Pizza, Restaurant                           17
Tacos, Restaurant                           16
Bar                                         16
Sushi, Restaurant                           14
Seafood, Restaurant                         13
Wine Bar, Bar                               13
Pub, Gastropub, Bar, Restaurant             13
Gastropub, Pub, Bar, Restaurant             12
Fine Dining, Restaurant                     11
Cocktail Bar, Bar                           11
Cafe                                        10
Wine Bar, Bar, Restaurant                    9
Sandwiches, Restaurant                       7
Breakfast, Restaurant                        7
Gastropub, Bar, Restaurant                   7
Restau

In [39]:
food[
    [
        "title",
        "primary_type",
        "cuisine",
        "format"
    ]
].sample(
    40,
    random_state=42
)

,title,primary_type,cuisine,format
421,Jenni’s Quesadillas,mexican_restaurant,Mexican,Restaurant
72,Ginza Barra,japanese_restaurant,Japanese,"Sushi, Restaurant"
528,The Rose & Crown,pub,None,"Pub, Bar, Restaurant"
73,Eno,restaurant,None,"Restaurant, Breakfast, Coffee Shop, Cafe"
932,The Ship,pub,None,"Pub, Beer Garden, Sports Bar, Bar, Restaurant"
836,Jolene Colebrooke Row,bakery,None,Bakery
92,La Bonvi - Virreyes,taco_restaurant,None,"Tacos, Restaurant"
877,Sophie's Soho,steak_house,None,"Steakhouse, Bar, Restaurant"
647,All My Friends,cocktail_bar,None,"Cocktail Bar, Bar, Restaurant"
497,DALLA,italian_restaurant,Italian,Restaurant


In [40]:
print(
    "Cuisine coverage:",
    food["cuisine"].notna().mean()
)

print(
    "Format coverage:",
    food["format"].notna().mean()
)

Cuisine coverage: 0.35517241379310344
Format coverage: 0.9908045977011494


In [41]:
missing_cuisine = food[
    food["cuisine"].isna()
].copy()

print(f"Missing cuisine: {len(missing_cuisine)}")

print("\nBY CATEGORY")
print(
    missing_cuisine["category"]
    .value_counts()
)

print("\nTOP PRIMARY TYPES")
print(
    missing_cuisine["primary_type"]
    .value_counts()
    .head(25)
)

Missing cuisine: 561

BY CATEGORY
category
Restaurant            240
Drinks & Nightlife    176
Coffee & Bakery       127
Food Shopping          11
Dessert                 7
Name: count, dtype: int64

TOP PRIMARY TYPES
primary_type
restaurant                155
pub                        94
coffee_shop                56
cafe                       37
bakery                     30
bar                        26
cocktail_bar               18
wine_bar                   17
gastropub                  15
taco_restaurant            12
pizza_restaurant           11
seafood_restaurant         10
sandwich_shop               8
bagel_shop                  8
ice_cream_shop              6
food_store                  6
brunch_restaurant           4
fine_dining_restaurant      4
irish_pub                   4
meal_takeaway               4
bistro                      3
steak_house                 3
market                      3
taiwanese_restaurant        2
night_club                  2
Name: count, dtype:

In [42]:
restaurants = food[
    food["category"] == "Restaurant"
].copy()

print(f"Total restaurants: {len(restaurants)}")

print(
    f"Restaurants with cuisine: "
    f"{restaurants['cuisine'].notna().sum()}"
)

print(
    f"Restaurants without cuisine: "
    f"{restaurants['cuisine'].isna().sum()}"
)

print(
    f"Cuisine coverage: "
    f"{restaurants['cuisine'].notna().mean():.1%}"
)

Total restaurants: 527
Restaurants with cuisine: 287
Restaurants without cuisine: 240
Cuisine coverage: 54.5%


### Food style

Cuisine coverage is incomplete because Google often describes restaurants by food type rather than cuisine. A broader `food_style` feature combines cuisine and food-specific restaurant types.

In [45]:
food_style_map = {
    # Cuisines
    "italian_restaurant": "Italian",
    "japanese_restaurant": "Japanese",
    "mexican_restaurant": "Mexican",
    "mediterranean_restaurant": "Mediterranean",
    "british_restaurant": "British",
    "french_restaurant": "French",
    "spanish_restaurant": "Spanish",
    "american_restaurant": "American",
    "indian_restaurant": "Indian",
    "chinese_restaurant": "Chinese",
    "asian_restaurant": "Asian",
    "thai_restaurant": "Thai",
    "greek_restaurant": "Greek",
    "lebanese_restaurant": "Lebanese",
    "australian_restaurant": "Australian",
    "middle_eastern_restaurant": "Middle Eastern",
    "basque_restaurant": "Basque",
    "latin_american_restaurant": "Latin American",
    "turkish_restaurant": "Turkish",
    "korean_restaurant": "Korean",
    "cantonese_restaurant": "Cantonese",
    "austrian_restaurant": "Austrian",
    "filipino_restaurant": "Filipino",
    "eastern_european_restaurant": "Eastern European",
    "persian_restaurant": "Persian",
    "peruvian_restaurant": "Peruvian",
    "polish_restaurant": "Polish",
    "german_restaurant": "German",
    "vietnamese_restaurant": "Vietnamese",
    "brazilian_restaurant": "Brazilian",
    "cajun_restaurant": "Cajun",
    "south_indian_restaurant": "South Indian",
    "caribbean_restaurant": "Caribbean",
    "south_american_restaurant": "South American",
    "taiwanese_restaurant": "Taiwanese",

    # Food styles
    "seafood_restaurant": "Seafood",
    "oyster_bar_restaurant": "Seafood",
    "steak_house": "Steakhouse",
    "vegetarian_restaurant": "Vegetarian",
    "vegan_restaurant": "Vegan",
    "halal_restaurant": "Halal",
    "barbecue_restaurant": "BBQ",
    "chicken_restaurant": "Chicken",

    # Strong cuisine signals from format
    "taco_restaurant": "Mexican",
    "sushi_restaurant": "Japanese",
    "ramen_restaurant": "Japanese",
    "japanese_curry_restaurant": "Japanese",
    "pizza_restaurant": "Italian",
    "hamburger_restaurant": "American",
    "falafel_restaurant": "Middle Eastern",
    "fish_and_chips_restaurant": "British",
    "chinese_noodle_restaurant": "Chinese",
}


def get_food_styles(types):

    if pd.isna(types):
        return None

    place_types = types.split(", ")

    styles = [
        food_style_map[t]
        for t in place_types
        if t in food_style_map
    ]

    if not styles:
        return None

    return ", ".join(dict.fromkeys(styles))


food["food_style"] = food[
    "google_types"
].apply(get_food_styles)



In [47]:
restaurants = food[
    food["category"] == "Restaurant"
].copy()

restaurants["food_style"] = restaurants[
    "google_types"
].apply(get_food_styles)

In [49]:
restaurants.columns.tolist()

['title',
 'note',
 'url',
 'tags',
 'comment',
 'source_list',
 'status',
 'google_place_id',
 'google_name',
 'google_maps_uri',
 'formatted_address',
 'neighborhood',
 'city',
 'state',
 'country',
 'country_code',
 'postal_code',
 'latitude',
 'longitude',
 'primary_type',
 'google_types',
 'business_status',
 'google_rating',
 'review_count',
 'price_level',
 'match_status',
 'category',
 'cuisine',
 'format',
 'food_style']

In [50]:
restaurants[
    "food_style"
].value_counts().head(30)

food_style
Italian                   63
Mexican                   50
Japanese                  36
British                   20
French                    16
Seafood                   15
Mediterranean             14
American                   9
Chinese                    8
Thai                       7
Indian                     7
Greek                      6
Spanish                    6
Italian, Mediterranean     4
Taiwanese                  3
Australian                 3
Persian                    2
French, Mediterranean      2
Asian                      2
Austrian                   2
Korean                     2
Lebanese                   2
Seafood, British           2
Basque, Spanish            2
Steakhouse                 2
Filipino                   2
Cantonese, Chinese         2
Seafood, French            2
Mediterranean, Spanish     1
Polish                     1
Name: count, dtype: int64

In [51]:
print(
    "Food style coverage:",
    f"{restaurants['food_style'].notna().mean():.1%}"
)

print(
    "Restaurants with food style:",
    restaurants["food_style"].notna().sum()
)

print(
    "Restaurants still missing:",
    restaurants["food_style"].isna().sum()
)

restaurants[
    "food_style"
].value_counts().head(30)

Food style coverage: 64.3%
Restaurants with food style: 339
Restaurants still missing: 188


food_style
Italian                   63
Mexican                   50
Japanese                  36
British                   20
French                    16
Seafood                   15
Mediterranean             14
American                   9
Chinese                    8
Thai                       7
Indian                     7
Greek                      6
Spanish                    6
Italian, Mediterranean     4
Taiwanese                  3
Australian                 3
Persian                    2
French, Mediterranean      2
Asian                      2
Austrian                   2
Korean                     2
Lebanese                   2
Seafood, British           2
Basque, Spanish            2
Steakhouse                 2
Filipino                   2
Cantonese, Chinese         2
Seafood, French            2
Mediterranean, Spanish     1
Polish                     1
Name: count, dtype: int64

### Restaurants with missing food style

Google place types identify a food style for 64.3% of restaurants. The remaining restaurants require additional information beyond structured place types.

In [52]:
missing_food_style = restaurants[
    restaurants["food_style"].isna()
].copy()

print(f"Restaurants to classify: {len(missing_food_style)}")

missing_food_style[
    [
        "title",
        "city",
        "neighborhood",
        "primary_type",
        "google_types",
        "google_rating",
        "review_count",
        "price_level"
    ]
].sample(
    30,
    random_state=42
)

Restaurants to classify: 188


,title,city,neighborhood,primary_type,google_types,google_rating,review_count,price_level
928,Little Duck The Picklery,Greater London,NaN,restaurant,"wine_bar, bar, restaurant, food, point_of_inte...",4.4,698.0,PRICE_LEVEL_EXPENSIVE
809,Oranj,Greater London,NaN,restaurant,"restaurant, wine_bar, cocktail_bar, coffee_sho...",4.0,389.0,PRICE_LEVEL_MODERATE
69,Máximo,Ciudad de México,Roma Norte,fine_dining_restaurant,"fine_dining_restaurant, restaurant, food, poin...",4.3,2819.0,PRICE_LEVEL_VERY_EXPENSIVE
54,Tacos Los Caramelos - Condesa,Ciudad de México,Colonia Condesa,restaurant,"restaurant, food, point_of_interest, establish...",4.8,1656.0,PRICE_LEVEL_MODERATE
332,Recio,Ciudad de México,San Miguel Chapultepec I Sección,restaurant,"restaurant, food, point_of_interest, establish...",5.0,17.0,PRICE_LEVEL_MODERATE
479,Sotto Enoteca & Trattoria,Edinburgh,NaN,restaurant,"restaurant, food, point_of_interest, establish...",4.6,214.0,PRICE_LEVEL_MODERATE
229,Cleo,Greater London,NaN,restaurant,"restaurant, brunch_restaurant, bar, food, poin...",4.4,412.0,PRICE_LEVEL_MODERATE
383,Izakalle,Ciudad de México,Roma Norte,restaurant,"restaurant, food, point_of_interest, establish...",4.9,176.0,PRICE_LEVEL_MODERATE
60,Marmota,Ciudad de México,Roma Norte,restaurant,"restaurant, food, point_of_interest, establish...",4.7,2144.0,NaN
704,The Cheesy Toast Shack,Fife,NaN,meal_takeaway,"meal_takeaway, fast_food_restaurant, restauran...",4.6,984.0,PRICE_LEVEL_INEXPENSIVE
